# Практика: Python

Укажите **ФИО**, **группу** и **кредит** → нажмите **Сгенерировать задание** → напишите решение в ячейке ниже → нажмите **Проверить**. Результат: оценка и комментарий (ошибки при наличии).

---
**Настройка (только для преподавателя).** Запустите следующую ячейку **один раз** перед началом работы. В Colab Secrets добавьте ключ `OPENAI_API_KEY`. Студентам переходить к ячейке с полями **ФИО, Группа, Кредит**.

In [ ]:
!pip install -q openai ipywidgets

from google.colab import userdata
from openai import OpenAI
import json
import hashlib
import ipywidgets as widgets
from IPython.display import display, clear_output

API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=API_KEY)
MODEL = 'gpt-4o-mini'

def generate_assignment(variant: int):
    prompt = f'''Сгенерируй ОДНО задание по программированию на Python для варианта {variant}. Требования: ввод/вывод (input/print), условные операторы (if/elif/else), сравнение строк, циклы (for/while), строки, списки, кортежи. Один связный сценарий, 15-25 строк кода. Язык: русский. Ответ — только текст задания, без кода.'''
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.7)
    return r.choices[0].message.content.strip()

def evaluate_solution(assignment_text: str, code: str, variant: int):
    prompt = f'''Задание (вариант {variant}):\n{assignment_text}\n\nКод студента:\n```python\n{code}\n```\nПроверь решение. Ответь СТРОГО JSON: {{\"score\": <0-100>, \"feedback\": \"комментарий на русском, при ошибках — опиши\"}}'''
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.3)
    text = r.choices[0].message.content.strip()
    if text.startswith('```'):
        text = text.split('```')[1].lstrip('json\n')
    try:
        d = json.loads(text)
        score = max(0, min(100, int(d.get('score', 0))))
        return {'score': score, 'feedback': d.get('feedback', '')}
    except Exception:
        return {'score': 0, 'feedback': 'Ошибка разбора ответа.'}

def save_to_sqlite(fio, group_name, credit, variant, score, feedback):
    try:
        import sqlite3
        import os
        if not os.path.exists('/content/drive'):
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
        path = '/content/drive/MyDrive/exam_results.db'
        conn = sqlite3.connect(path)
        conn.execute('''CREATE TABLE IF NOT EXISTS results (fio TEXT, group_name TEXT, credit TEXT, variant INT, score INT, feedback TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
        conn.execute('INSERT INTO results (fio, group_name, credit, variant, score, feedback) VALUES (?,?,?,?,?,?)', (fio, group_name, credit, variant, score, (feedback or '')[:2000]))
        conn.commit()
        conn.close()
    except Exception:
        pass

current_assignment = ''
current_variant = 1
current_fio = ''
current_group = ''
current_credit = ''
print('Готово.')

---
## ФИО, группа, кредит

In [ ]:
fio_in = widgets.Text(placeholder="ФИО", description="ФИО:", style={"description_width": "60px"})
group_in = widgets.Text(placeholder="Группа", description="Группа:", style={"description_width": "60px"})
credit_in = widgets.Text(placeholder="Кредит", description="Кредит:", style={"description_width": "60px"})
out_assign = widgets.Output()

def on_generate(b):
    global current_assignment, current_variant, current_fio, current_group, current_credit
    fio, group, credit = fio_in.value.strip(), group_in.value.strip(), credit_in.value.strip()
    if not fio or not group or not credit:
        with out_assign:
            clear_output(wait=True)
            print("Заполните ФИО, группу и кредит.")
        return
    h = int(hashlib.sha256(f"{credit}{group}{fio}".encode()).hexdigest(), 16) % 17
    current_variant = h + 1
    current_fio, current_group, current_credit = fio, group, credit
    with out_assign:
        clear_output(wait=True)
        print("Генерация задания...")
    current_assignment = generate_assignment(current_variant)
    with out_assign:
        clear_output(wait=True)
        print(f"Вариант {current_variant}")
        print("-" * 40)
        print(current_assignment)

btn_gen = widgets.Button(description="Сгенерировать задание", button_style="primary")
btn_gen.on_click(on_generate)
display(widgets.VBox([fio_in, group_in, credit_in, btn_gen, out_assign]))

Напишите решение в ячейке ниже (переменная `student_code`) и **запустите ячейку**, затем нажмите **Проверить**.

In [ ]:
student_code = """
# Ваше решение
pass
"""

## Проверить задание

In [ ]:
out_check = widgets.Output()

def on_check(b):
    global current_assignment, current_variant, current_fio, current_group, current_credit
    with out_check:
        clear_output(wait=True)
        print("Проверка...")
    code = ""
    try:
        code = get_ipython().user_ns.get("student_code", "")
    except Exception:
        code = ""
    if not code or code.strip() in ("", "pass", "# Ваше решение"):
        with out_check:
            clear_output(wait=True)
            print("Сначала напишите решение в ячейке выше и запустите её.")
        return
    if not current_assignment:
        with out_check:
            clear_output(wait=True)
            print("Сначала нажмите «Сгенерировать задание».")
        return
    result = evaluate_solution(current_assignment, code, current_variant)
    try:
        save_to_sqlite(current_fio, current_group, current_credit, current_variant, result["score"], result["feedback"])
    except Exception:
        pass
    with out_check:
        clear_output(wait=True)
        print(f"Оценка: {result['score']} / 100")
        print(f"Комментарий (ошибки при наличии): {result['feedback']}")

btn_check = widgets.Button(description="Проверить", button_style="success")
btn_check.on_click(on_check)
display(widgets.VBox([btn_check, out_check]))